# Implementing `Co-Clustering Triples from Open Information Extraction" by Pal, Ho, and Weikum` From Scratch

## 1. Motivation and Problem Statement
At core of this paper it tackles the "messiness" of information extracted from raw text all over the internet.<br>
Imagine we have systems called Open Information Extraction (Open IE) that read billions of sentences from the web and try to distill them into simple facts. These facts are stored as triples: (Subject, Predicate, Object), or (SPO). 
For example, `New Delhi is India's capital` becomes $(New Delhi, is capital of, India).$

This process is powerful but it also ends up creating two major problems:

- **Redundancy and Ambiguity**:  
  Natural human language is flexible therefore the same fact can be written in many ways. For example, `(Apple, was founded by, Steve Jobs)` and `(Apple, was started by, Steve Jobs)` are semantically identical. This is **redundancy**.  
  Conversely, a single predicate can have completely different meanings depending on the context.  
  For example consider the triples `(lion, attack, zebra)` versus `(student, attack, problem)`.  
  The predicate "attack" here is used in two vastly different senses i.e. **ambiguity**[.

- **Predicate Imbalance**:  
  A few generic verbs like "is a", "have", and "include" appear often covering a huge percentage of all extracted triples.  
  Whereas at the same time there's a "long tail" of highly specific and rare predicates that appear only a handful of times.  
  The generic predicates often tell nothing clearly about the true semantic relationship while the rare ones are too infrequent for most algorithms to learn from.

The core problem the paper addresses is the lack of **canonicalization**

> Adding to these issues is the problem of exact mismatch where triples may be semantically similar but represent different levels of specificity. For example when we say the fact (panda, are, endangered) is a general statement while (panda, are, endangered in China) is a more specific refinement. An ideal canonicalization system must be able to recognize the relationship between such statements.

## 2. Assumptions, Goals, and Research Questions

In response to the identified shortcomings of previous methods, Pal et al. propose a novel framework centered on the principles of soft co-clustering. The approach is designed not merely to group synonymous phrases but to perform a more sophisticated context-aware canonicalization that can handle the nuances of natural language as found in OIE triples.

### 2.1 The Core Idea: Jointly Clustering Predicates and Arguments

The central hypothesis of the paper is that the ambiguity of predicates can only be resolved by considering the semantic context provided by their arguments. Hence instead of clustering predicates and subject-object(SO) pairs in isolation the proposed method clusters them jointly and learns the alignment between these clusters. This concept is best illustrated by the paper's own example:

Given a set of triples including `(student, attack, problem)`, `(researcher, solve, problem)`, `(lion, attack, zebra)`, and `(lion, kill, zebra)` the desired output is not a single cluster for the predicate "attack." Instead of this the ideal output would be two distinct co-clusters:
1. A predicate cluster `{attack, solve}` aligned with an SO-pair cluster `{student-problem, researcher-problem}`.
2. A predicate cluster `{kill, attack}` aligned with an SO-pair cluster `{lion-zebra, lion-antelope}`.

This example perfectly explains the power of the co-clustering paradigm. It correctly differentiates the two senses of "attack" by aligning them with two semantically coherent, but distinct groups of arguments. This joint approach is the cornerstone of the paper's methodology which is designed to capture the interplay between what is being said (the predicate) and what it is being said about (the arguments).

### 2.2 The Three Goal of the Framework
The authors articulate three specific, interconnected goals that their soft-co-clustering framework aims to achieve: Normalization, Specialization, and Transfer Learning.

1. **Normalization:** This is the traditional goal of canonicalization i.e. to focus on reducing redundancy. It involves grouping together different predicate phrases that are semantically equivalent. For example, the model should learn to group predicates like "possess," "control," and "occupy" into a single conceptual cluster representing ownership or something dominant. This addresses the many-to-one mapping from various linguistic surface forms to a single underlying meaning.

2. **Specialization:** This is the paper's most significant and novel conceptual contribution. Specialization addresses the problem of polysemy i.e. the coexistence of many possible meanings for a word or phrase by forming context-specific clusters.<br>
It allows a single predicate to be assigned to multiple different clusters where each represent a specific sense or usage determined by its arguments. The paper here is using predicate "have" as an example. Depending on the SO-pairs it co-occurs with, "have" might be grouped with possessive predicates like "has part" (e.g., a car "has" an engine) or with biological predicates like "grow" and "form" (e.g., a tree "has" leaves). While Normalization is about finding synonyms specialization is used to find distinct senses.<br>
This addresses the one-to-many mapping from a single surface form to multiple context-dependent meanings i.e. a more sophisticated and realistic model of language that hard-clustering methods are fundamentally incapable of handling.

3. **Transfer Learning:** This goal highlights a beneficial side-effect of the clustering process demonstrating its potential for knowledge discovery similaar to knowledge base completion. By grouping entities and predicates based on semantic similarity the model can generalize learned patterns to derive new plausible facts. The paper's example is illustrative: i.e. if the model learns the fact `(lion, attack, zebra)` and also learns to cluster hyena with lion (because they appear in similar SO-pair contexts) it can potentially infer the new fact `(hyena, attack, zebra)`. This showcases the framework's ability not just to organize existing knowledge but also it can make it better.

### 2.3 Why Soft Clustering is Essential
The goal of Specialization makes the choice of a ***soft*** clustering mechanism non-negotiable. As seen in the past works that relied on "hard clustering" i.e. assigning each item to a single exclusive cluster is a major limitation when dealing with the polysemy of natural language.

Whereas soft clustering allows an item (a predicate or an SO-pair) to hold partial membership in multiple clusters at same time typically represented by a vector of likelihoods or probabilities. This is precisely the mechanism needed to enable Specialization. A predicate like "attack" can have a high membership score in the "predator-prey" cluster and at the same time a non-zero membership score in the "academic problem-solving" cluster. This allows the model to capture the multiple domain of a word's meaning without being forced into an wrong all-or-nothing decision.

## 3. The Mathematical Behind Idea of Tri-Factorization

To implement their soft-co-clustering framework the authors of paper adapt a powerful mathematical tool: **Non-Negative Matrix Tri-Factorization (NMTF).** This section provides a technically deep analysis of the model's architecture from its foundational principles to its specific optimization landscape.

### 3.1 Foundation with Matrix Factorization for Latent Semantics
At its base matrix factorization (MF) is a class of algorithms used in linear algebra and machine learning to decompose a matrix into a product of two or more lower-rank matrices. The central intuition is that these lower-rank "factor" matrices can reveal hidden structures or meaning within the original data.

A classic example is in recommender systems. Imagine a large matrix where rows represent users and columns represent movies with the entries being the ratings users have given. This matrix is likely very sparse (most users haven't rated most movies). MF decomposes this user-movie matrix into two smaller and denser matrices: a user-feature matrix and a movie-feature matrix.<br>
The user-feature matrix might learn hidden features for each user (e.g., preference for comedy, action, drama) while the movie-feature matrix learns corresponding hidden features or genre for each movie (e.g., degree of comedy, action, drama). By multiplying these factor matrices back together the model can predict the ratings for the missing entries effectively recommending movies to users. The key takeaway is that MF is a technique for discovering unobserved hidden dimensions that explain the observed interactions in the data. This principle is fundamental to understanding the  $U$, $W$, and $V$ matrices in the paper's model.

### 3.2 The Non-Negative Matrix Tri-Factorization (NMTF) Model

The paper employs a specific variant of Matrix Factorisation called Non-Negative Matrix Tri-Factorization. This approach is particularly well-suited for the task due to its interpretability and its ability to model three-way interactions.

#### 3.2.1 Constructing the Input Matrix $M$